In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from pydantic import BaseModel, Field
from IPython.display import Image, display

In [ ]:
chat_template_players = ChatPromptTemplate.from_template(
    '''
    Suggest three legendary {football position} players in football history.
    Answer only by listing the player names.
    '''
)

chat_template_matches = ChatPromptTemplate.from_template(
    '''
    Suggest three of the most memorable FIFA World Cup matches involving {team}.
    Answer only by listing the matches.
    '''
)

In [ ]:
class LegendaryPlayers(BaseModel):
    """Three legendary football players for a given position."""

    players: list[str] = Field(
        min_length=3,
        max_length=3,
        description="Three legendary players for the position",
    )


class MemorableMatches(BaseModel):
    """Three memorable FIFA World Cup matches for a given national team."""

    matches: list[str] = Field(
        min_length=3,
        max_length=3,
        description="Three memorable World Cup matches involving the team",
    )

In [ ]:
llm = ChatLiteLLM(
    model="gpt-5.4-mini",
    temperature=0,
    model_kwargs={'seed': 365},
    max_tokens=100,
)

In [ ]:
llm_players = llm.with_structured_output(LegendaryPlayers)
llm_matches = llm.with_structured_output(MemorableMatches)

In [ ]:
chain_players = chat_template_players | llm_players
chain_matches = chat_template_matches | llm_matches

In [ ]:
chain_parallel = RunnableParallel({'players': chain_players, 'matches': chain_matches})

In [ ]:
chain_parallel.invoke({'football position': 'striker', 'team': 'Brazil'})

In [ ]:
display(Image(chain_parallel.get_graph().draw_mermaid_png()))

In [ ]:
%%time
chain_players.invoke({'football position': 'striker', 'team': 'Brazil'})

In [ ]:
%%time
chain_matches.invoke({'football position': 'striker', 'team': 'Brazil'})

In [ ]:
%%time
chain_parallel.invoke({'football position': 'striker', 'team': 'Brazil'})